<a href="https://colab.research.google.com/github/kartikirasane27/Deep_Learning/blob/main/Assignment9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torchvision -q

In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np

from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet18, ResNet18_Weights
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

classes = test_dataset.classes

subset = Subset(test_dataset, range(100))
loader = DataLoader(subset, batch_size=16, shuffle=False)

print("Classes:", classes)
print("Number of test images:", len(subset))

100%|██████████| 170M/170M [58:15<00:00, 48.8kB/s]


Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Number of test images: 100


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

vit_processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

vit_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224"
).to(device)

vit_model.eval()

Device: cpu


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [5]:
cnn_model = resnet18(weights=ResNet18_Weights.DEFAULT).to(device)
cnn_model.eval()

cnn_transform = ResNet18_Weights.DEFAULT.transforms()

print("ViT and CNN models loaded successfully!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 59.6MB/s]


ViT and CNN models loaded successfully!


In [6]:
vit_correct = 0
vit_total = 0
vit_start = time.time()

for images, labels in loader:
    pil_images = [
        transforms.ToPILImage()(img) for img in images
    ]

    inputs = vit_processor(
        images=pil_images,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = vit_model(**inputs)

    predictions = outputs.logits.argmax(dim=1)

    for pred, label in zip(predictions, labels):
        predicted_name = vit_model.config.id2label[pred.item()].lower()
        actual_name = classes[label.item()]

        if actual_name in predicted_name:
            vit_correct += 1

    vit_total += len(labels)

vit_time = time.time() - vit_start
vit_accuracy = 100 * vit_correct / vit_total

print("ViT Accuracy:", round(vit_accuracy, 2), "%")
print("ViT Execution Time:", round(vit_time, 2), "seconds")

ViT Accuracy: 24.0 %
ViT Execution Time: 75.15 seconds


In [7]:
cnn_correct = 0
cnn_total = 0
cnn_start = time.time()

for images, labels in loader:
    images = torch.stack([
        cnn_transform(transforms.ToPILImage()(img))
        for img in images
    ]).to(device)

    with torch.no_grad():
        outputs = cnn_model(images)

    predictions = outputs.argmax(dim=1)

    for pred, label in zip(predictions, labels):
        predicted_name = cnn_model.fc
        actual_name = classes[label.item()]

        # ImageNet class names are checked below.
        from torchvision.models import ResNet18_Weights
        imagenet_classes = ResNet18_Weights.DEFAULT.meta["categories"]
        name = imagenet_classes[pred.item()].lower()

        if actual_name in name:
            cnn_correct += 1

    cnn_total += len(labels)

cnn_time = time.time() - cnn_start
cnn_accuracy = 100 * cnn_correct / cnn_total

print("CNN Accuracy:", round(cnn_accuracy, 2), "%")
print("CNN Execution Time:", round(cnn_time, 2), "seconds")

CNN Accuracy: 4.0 %
CNN Execution Time: 16.27 seconds


In [8]:
print("\nPerformance Comparison")
print("-" * 40)

print("Model       Accuracy (%)    Time (seconds)")
print(f"ViT         {vit_accuracy:.2f}          {vit_time:.2f}")
print(f"CNN         {cnn_accuracy:.2f}          {cnn_time:.2f}")

if vit_accuracy > cnn_accuracy:
    print("\nViT achieved higher accuracy.")
elif cnn_accuracy > vit_accuracy:
    print("\nCNN achieved higher accuracy.")
else:
    print("\nBoth models achieved equal accuracy.")


Performance Comparison
----------------------------------------
Model       Accuracy (%)    Time (seconds)
ViT         24.00          75.15
CNN         4.00          16.27

ViT achieved higher accuracy.
